# Data collection

Imports

In [1]:
import pandas as pd
from pathlib import Path
import yfinance as yf

### Step 1: Convert historical membership data to historical tickers dataframe & store in one file.

In [2]:
data_folder_path = Path("data")
membership_snapshots_path = data_folder_path / "membership_snapshots"

# Create the folder and its parents if they don't exist
membership_snapshots_path.mkdir(parents=True, exist_ok=True)

print(membership_snapshots_path.exists())

True


In [3]:
series_list = []

# The files have no extension
for file_path in sorted(membership_snapshots_path.glob("*")):
    s = pd.read_csv(
        file_path,
        sep=r"\t+",
        engine="python",
        header=None,
        usecols=[0]
    )[0]

    s = s.str.strip().str.replace('/', '-', regex=False) # fix ticker formatting to match yfinance format

    s.name = file_path.stem[3:] # remove the "spy" prefix, leaving the name as YYYYMM
    series_list.append(s)

membership_df = pd.concat(series_list, axis=1)

In [4]:
print(membership_df.head())

  201601 201602 201603 201604 201605 201606 201607 201608 201609 201610  ...  \
0      A      A      A      A      A      A      A      A      A      A  ...   
1     AA     AA     AA     AA     AA     AA     AA     AA     AA     AA  ...   
2    AAL    AAL    AAL    AAL    AAL    AAL    AAL    AAL    AAL    AAL  ...   
3    AAP    AAP    AAP    AAP    AAP    AAP    AAP    AAP    AAP    AAP  ...   
4   AAPL   AAPL   AAPL   AAPL   AAPL   AAPL   AAPL   AAPL   AAPL   AAPL  ...   

  202507 202508 202509 202510 202511 202512 202601 202602 202603 202604  
0      A      A      A      A      A      A      A      A      A      A  
1   AAPL   AAPL   AAPL   AAPL   AAPL   AAPL   AAPL   AAPL   AAPL   AAPL  
2   ABBV   ABBV   ABBV   ABBV   ABBV   ABBV   ABBV   ABBV   ABBV   ABBV  
3   ABNB   ABNB   ABNB   ABNB   ABNB   ABNB   ABNB   ABNB   ABNB   ABNB  
4    ABT    ABT    ABT    ABT    ABT    ABT    ABT    ABT    ABT    ABT  

[5 rows x 124 columns]


Save to CSV

In [5]:
membership_df.to_csv(data_folder_path / "membership_df.csv", index=False)

### Step 2: Store historical OHLCV price data

Get all unique tickers from January 2016 to January 2026

In [6]:
unique_tickers = pd.Series(membership_df.values.ravel()).dropna().unique().tolist()
print(unique_tickers)

['A', 'AA', 'AAL', 'AAPL', 'AAP', 'ABBV', 'ABNB', 'ABT', 'ABMD', 'ACGL', 'ACN', 'ADBE', 'ADI', 'ADM', 'ADP', 'ADS', 'ADSK', 'AEE', 'AEP', 'ADT', 'AES', 'AFL', 'AET', 'AIG', 'AIZ', 'AGN', 'AIV', 'AJG', 'AKAM', 'ALB', 'ALGN', 'ALK', 'ALL', 'ALLE', 'AMAT', 'AMCR', 'ALXN', 'AMD', 'AME', 'AMGN', 'AMP', 'AMG', 'AMT', 'AMTM', 'AMZN', 'ANET', 'ANSS', 'AON', 'AOS', 'AN', 'APA', 'ANDV', 'APD', 'APH', 'APO', 'APC', 'APTV', 'APP', 'ARE', 'ATO', 'AVB', 'ARES', 'ARG', 'ATVI', 'AVGO', 'ARNC', 'AVY', 'AWK', 'AXON', 'AXP', 'AZO', 'BA', 'AYI', 'BAC', 'BALL', 'BAX', 'BBY', 'BBBY', 'BBWI', 'BDX', 'BEN', 'BF-B', 'BCR', 'BG', 'BIIB', 'BK', 'BKNG', 'BHI', 'BIO', 'BKR', 'BLDR', 'BLK', 'BMY', 'BHF', 'BR', 'BRK-B', 'BRCM', 'BRO', 'BSX', 'BWA', 'BX', 'BXLT', 'BXP', 'C', 'CAG', 'CA', 'CAH', 'CARR', 'CAT', 'CAM', 'CB', 'CBOE', 'CBRE', 'CCI', 'CCE', 'CCL', 'CDNS', 'CELG', 'CDW', 'CERN', 'CEG', 'CF', 'CE', 'CFG', 'CHD', 'CHK', 'CHRW', 'CHTR', 'CI', 'CINF', 'CL', 'CLX', 'CIEN', 'CMA', 'CMCSA', 'CME', 'CMG', 'CMI', 'C

Pull data for these tickers from January 2014 to January 2026 (2 extra years as a loopback period)

In [7]:
start_date = "2014-01-01"
end_date = "2026-01-01"

In [8]:
historical_ohlcv_df = yf.download(unique_tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=True) # auto adjust=True to account for splits and dividends

[                       0%                       ]$SNI: possibly delisted; no timezone found
$LLL: possibly delisted; no timezone found
[                       1%                       ]  4 of 705 completed$WBA: possibly delisted; no timezone found
[**                     4%                       ]  29 of 705 completed$PXD: possibly delisted; no timezone found
[**                     5%                       ]  32 of 705 completed$TSS: possibly delisted; no timezone found
[**                     5%                       ]  35 of 705 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CMA"}}}
[**                     5%                       ]  36 of 705 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: LM"}}}
[***                    7%                       ]  48 of 705 completed$CMA: possibly delisted; no timezone found
[***            

In [9]:
print(historical_ohlcv_df.head())

Ticker     SOLV                       KVUE                        ...  \
Price      Open High Low Close Volume Open High Low Close Volume  ...   
Date                                                              ...   
2014-01-02  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   
2014-01-03  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   
2014-01-06  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   
2014-01-07  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   
2014-01-08  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   

Ticker              R                                                 EOG  \
Price            Open       High        Low      Close  Volume       Open   
Date                                                                        
2014-01-02  51.244708  51.384891  50.683965  50.915272  453500  59.113503   
2014-01-03  51.055471  51.335842  50.838185  51.020428  303500  58.822462   
2014-01-06  51.265752  51.4479

Save to parquet file (good for handling MultiIndex colums)

In [10]:
historical_ohlcv_df.to_parquet(
    data_folder_path / "historical_ohlcv_df.parquet"
)